# AWS Open-Data S3 — catalog explorer

No network needed: list the registered datasets, inspect a dataset's metadata,
resolve variables, and dry-run the key planner. Everything here reads from the
bundled S3 catalog — nothing is downloaded.

## Setup

`EarthLens` is the unified entry point; `Catalog` is the S3 catalog loader, and
the `S3` backend exposes the list of registered datasets directly.

In [ ]:
from earthlens.core import EarthLens
from earthlens.s3 import S3, Catalog

## Registered datasets

`S3.datasets()` returns the keys of every AWS Open-Data dataset the backend
knows about — the names you can pass as `dataset=`.

In [ ]:
S3.datasets()

## Inspect a dataset

Load the catalog and resolve the `era5` row, then print its bucket, on-disk
format, and CRS alongside the variables it exposes.

In [ ]:
cat = Catalog()
era5 = cat.resolve('era5')
print('bucket:', era5.bucket, '| format:', era5.format, '| crs:', era5.crs)
print('default vars:', era5.default_variables)
sorted(era5.variables)

## Resolve variables by any name

Variables resolve by friendly name, alias, **or** raw native token — all to one
row, so you can request data with whichever spelling you have on hand.

In [ ]:
for key in ['t2m', '2m_temperature', '128_167_2t']:
    print(key, '->', era5.resolve_variable(key).native)

## Dry-run the key planner

Build a request the normal way and call `_search()` to plan the S3 keys it would
fetch — without downloading anything. Here we ask for a one-day
`copernicus-dem` window over a small AOI.

In [ ]:
src = EarthLens(
    data_source="amazon-s3",
    start='2021-01-01',
    end='2021-01-01',
    aoi=[6.4, 0.4, 6.6, 0.6],
    dataset='copernicus-dem',
)

Iterate the planned hrefs to see exactly which objects the request resolves to.

In [ ]:
for p in src._search():
    print(p.href)